## PER2526, grupo de laboratorio L1-3CO11
## A3. Prueba práctica de laboratorio de B2 (cuaderno) (1.25 puntos)
## 26 de mayo de 2026

**Ejercicio:** $\;$ Suponiendo que queremos adaptar el modelo "resnet50.fb_swsl_ig1b_ft_in1k" a la tarea Cifar-10 siguiendo los pasos realizados en las sesiones de prácticas, completa el siguiente cuaderno para preparar un experimento en que se realize un finetuning. En el experimento, entrenaremos el modelo durante 5 épocas utilizando una transformación de reflejo horizontal aleatorio con una probabilidad de `0.7`, además de un scheduler `CosineAnnealing` con un learning rate minimo de `1e-6`.

**Nota:** $\;$ Antes de empezar el experimento, debes introducir las **últimas 3** cifras de tu DNI/NIE en el campo `dni` de la segunda celda del cuaderno

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import timm
from torchvision.transforms import v2
from torch.utils.data import DataLoader
import datasets

In [ ]:
dni=515 # Completar con tu DNI

In [ ]:
def exp(model, device, train_loader, test_loader, optimizer, scheduler, epochs=15):
    loss_fn = torch.nn.CrossEntropyLoss()
    for epoch in range(epochs):
        print(f'Epoch {epoch}: train:', end=' ')
        model.train(); trsize, trbatches, trloss, tracc = 0, 0, 0, 0
        for batch in train_loader:
            X, y = batch['X'].to(device), batch['y'].to(device); trsize += len(X); trbatches += 1
            pred = model(X); loss = loss_fn(pred, y); trloss += loss.item()
            tracc += (pred.argmax(1) == y).type(torch.float).sum().item()
            loss.backward(); optimizer.step(); optimizer.zero_grad(); scheduler.step()
        trloss /= trbatches; tracc /= trsize
        print(f'loss {trloss:g} acc {tracc:.2%} test:', end=' ')
        model.eval(); tesize, tebatches, teloss, teacc = 0, 0, 0, 0
        with torch.no_grad():
            for batch in test_loader:
                X, y = batch['X'].to(device), batch['y'].to(device); tesize += len(X); tebatches += 1
                pred = model(X); teloss += loss_fn(pred, y).item()
                teacc += (pred.argmax(1) == y).type(torch.float).sum().item()
        teloss /= tebatches; teacc /= tesize
        print(f'loss {teloss:g} acc {teacc:.2%}')

In [ ]:
ds = datasets.load_dataset("uoft-cs/cifar10").rename_columns({'img': 'X', 'label': 'y'})
train_ds = ds['train'].to_iterable_dataset(num_shards=1024).shuffle(seed=dni)
test_ds = ds['test'].to_iterable_dataset(num_shards=1024)

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
model_name = "resnet50.fb_swsl_ig1b_ft_in1k"
model = timm.create_model(None).to(device) # Completar

In [ ]:
data_cfg = timm.data.resolve_data_config(model.pretrained_cfg)
data_cfg['input_size'] = None # Completar
timm.data.create_transform(**data_cfg)

In [ ]:
train_transform = v2.Compose([
    None]) # Completar
def train_prep(example):
    return { 'X' : train_transform(example['X']), 'y' : example['y']}
train_ds = train_ds.map(train_prep, batched=True)
train_loader = DataLoader(train_ds, batch_size=32)

test_transform = v2.Compose([
    None]) # Completar
def test_prep(example):
    return { 'X' : test_transform(example['X']), 'y' : example['y']}
test_ds = test_ds.map(test_prep, batched=True)
test_loader = DataLoader(test_ds, batch_size=32)

In [ ]:
for p in model.parameters():
    p.requires_grad = False

for p in model.fc.parameters():
    p.requires_grad = True

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3) 
steps = 5
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(None) # Completar

exp(model, device, train_loader, test_loader, optimizer, scheduler, epochs=steps)


<p style="page-break-after:always;"></p>
